# Modern Data Science in Python: Getting Started

This notebook verifies that your environment is set up correctly and gives you a quick taste of the four main packages we'll use in this workshop:

- **Polars** — fast data manipulation
- **Plotnine** — Grammar of Graphics for Python
- **Great Tables** — presentation-ready tables
- **scikit-learn** — machine learning

We'll use the Palmer Penguins dataset throughout.

## Load the data with Polars

Polars is a lightning-fast DataFrame library. Here we load the penguins dataset from CSV and take a first look.

In [ ]:
import polars as pl
from palmerpenguins import load_penguins

penguins = pl.from_pandas(load_penguins()).drop_nulls()
penguins.head()

## Data manipulation with Polars

Polars uses a method-chaining style that will feel familiar to dplyr users. Let's compute the average body mass by species and sex.

In [ ]:
summary = (
    penguins
    .group_by("species", "sex")
    .agg(
        pl.col("body_mass_g").mean().alias("avg_mass_g"),
        pl.col("body_mass_g").count().alias("count"),
    )
    .sort("species", "sex")
)
summary

## Visualization with Plotnine

Plotnine implements the Grammar of Graphics (like ggplot2 in R). Let's create a scatter plot of bill length versus bill depth, colored by species.

In [ ]:
from plotnine import *

(
    ggplot(penguins, aes(x="bill_length_mm", y="bill_depth_mm", color="species"))
    + geom_point(size=2, alpha=0.7)
    + labs(
        title="Penguin Bill Dimensions",
        x="Bill Length (mm)",
        y="Bill Depth (mm)",
    )
)

## Presentation tables with Great Tables

Great Tables turns DataFrames into polished, publication-ready tables. Let's display our summary from earlier.

In [ ]:
from great_tables import GT, md

(
    GT(summary)
    .tab_header(
        title="Average Penguin Body Mass",
        subtitle="Grouped by species and sex",
    )
    .fmt_number("avg_mass_g", decimals=0)
    .cols_label(
        species="Species",
        sex="Sex",
        avg_mass_g=md("Avg Mass *(g)*"),
        count="Count",
    )
)

## Machine learning with scikit-learn

Scikit-learn provides a consistent API for building ML models. Let's train a simple classifier to predict penguin species from their measurements.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

X = penguins.select(feature_cols).to_numpy()
y = penguins.get_column("species").to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}")